# Per-Group TARP Calibration — DiffDock

Investigates DiffDock's calibration **separately** for each of its three diffusion groups:
- **Translation** (R³): where does the ligand centroid land?
- **Rotation** (SO(3)): how is the ligand oriented?
- **Torsion** (T^k): what are the internal dihedral angles?

Uses TARP (Lemos et al., ICML 2023): ECP(α) vs α curves, where perfect calibration = diagonal,
curve below diagonal = overconfident (posterior too narrow), above = underconfident.

Data computed by `analysis/run_group_eval.py` submitted via `~/slurm/diffdock_group_eval.sh`.

## 0 · Setup

In [ ]:
import sys, os, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

from utils.tarp_eval import ecp_from_fractions, plot_ecp, bootstrap_ecp

# ── Paths ─────────────────────────────────────────────────────────────────────
PDB_METRICS   = "/home/qf226/rds/results/pdbbind_testset/metrics"
PDB_GE        = os.path.join(PDB_METRICS, "group_eval")
PB_METRICS    = "../results/posebusters_inference/metrics"
PB_GE         = os.path.join(PB_METRICS, "group_eval")
FIGURES       = os.path.join(PDB_METRICS, "..", "figures")
os.makedirs(FIGURES, exist_ok=True)

GROUPS  = ['translation', 'rotation', 'torsion']
COLORS  = {'translation': 'C0', 'rotation': 'C1', 'torsion': 'C2'}
LABELS  = {'translation': 'Translation (R³)', 'rotation': 'Rotation (SO(3))', 'torsion': 'Torsion (T^k)'}
UNITS   = {'translation': 'Å', 'rotation': 'rad', 'torsion': 'rad (RMS)'}

def load_npy(path, allow_pickle=False):
    """Load .npy returning None if file doesn't exist."""
    if os.path.exists(path):
        return np.load(path, allow_pickle=allow_pickle)
    print(f'  [missing] {path}')
    return None

print('Setup complete.')

## 1 · Load Data

In [ ]:
# ── PDBBind group eval outputs ─────────────────────────────────────────────────
pdb_names_tarp = load_npy(f'{PDB_GE}/complex_names.npy', allow_pickle=True)
pdb_names_dist = load_npy(f'{PDB_GE}/complex_names_distances.npy', allow_pickle=True)
pdb_nrot       = load_npy(f'{PDB_GE}/n_rot_bonds.npy')
pdb_nrot_tarp  = load_npy(f'{PDB_GE}/n_rot_bonds_tarp.npy')

pdb_tarp = {}
pdb_dist = {}
for g in GROUPS:
    pdb_tarp[g] = load_npy(f'{PDB_GE}/tarp_fractions_{g}.npy')
for g in ['translation', 'rotation', 'torsion_rms']:
    pdb_dist[g] = load_npy(f'{PDB_GE}/distances_{g}.npy')

# ── PoseBusters group eval outputs ────────────────────────────────────────────
pb_names_tarp = load_npy(f'{PB_GE}/complex_names.npy', allow_pickle=True)
pb_names_dist = load_npy(f'{PB_GE}/complex_names_distances.npy', allow_pickle=True)
pb_nrot       = load_npy(f'{PB_GE}/n_rot_bonds.npy')
pb_nrot_tarp  = load_npy(f'{PB_GE}/n_rot_bonds_tarp.npy')

pb_tarp = {}
pb_dist = {}
for g in GROUPS:
    pb_tarp[g] = load_npy(f'{PB_GE}/tarp_fractions_{g}.npy')
for g in ['translation', 'rotation', 'torsion_rms']:
    pb_dist[g] = load_npy(f'{PB_GE}/distances_{g}.npy')

# ── Existing aggregated metrics (PDBBind) ─────────────────────────────────────
pdb_top1_rmsd  = load_npy(f'{PDB_METRICS}/top1_rmsd.npy')
pdb_all_names  = load_npy(f'{PDB_METRICS}/complex_names.npy', allow_pickle=True)

# ── PoseBusters validity results (PDBBind set) ────────────────────────────────
import json
_pb_path = f'{PDB_METRICS}/posebusters_results.json'
pb_validity = json.load(open(_pb_path)) if os.path.exists(_pb_path) else {}

print(f'PDBBind  TARP: {pdb_names_tarp is not None and len(pdb_names_tarp)} complexes')
print(f'PDBBind  dist: {pdb_names_dist is not None and len(pdb_names_dist)} complexes')
print(f'PoseBusters TARP: {pb_names_tarp is not None and len(pb_names_tarp)} complexes')
print(f'PoseBusters dist: {pb_names_dist is not None and len(pb_names_dist)} complexes')

## 2 · Per-Group TARP ECP Curves

TARP (Tests of Accuracy with Random Points, Lemos et al. 2023):
- For each complex, draw K random reference points from the prior, compute distances to the crystal and all samples
- f(reference, complex) = fraction of samples closer to crystal than the reference
- ECP(α) = fraction of complexes where f ≤ α
- Perfect calibration: ECP = α (diagonal). Below diagonal = overconfident. Above = underconfident.

**Prior distributions used:**
- Translation: N(Cα COM, σ²I) where σ = std_ca × 1.46 / 1.73 (DiffDock's prior at t=1)
- Rotation: Haar-uniform on SO(3) (random unit quaternion)
- Torsion: Uniform([-π, π])^k

In [ ]:
# 2a: Three-panel ECP — PDBBind
if pdb_names_tarp is not None and all(pdb_tarp[g] is not None for g in GROUPS):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    for ax, g in zip(axes, GROUPS):
        f = pdb_tarp[g]
        ecp, alpha = ecp_from_fractions(f)
        boot = bootstrap_ecp(f, n_bootstrap=500)
        n_valid = np.sum(np.any(np.isfinite(f), axis=1))
        plot_ecp(ecp, alpha, ax=ax, label=f'{LABELS[g]}\n(n={n_valid})', color=COLORS[g], bootstrap_ecps=boot)
        ax.set_title(LABELS[g], fontsize=11)
    fig.suptitle('Per-group TARP ECP — DiffDock on PDBBind test set (n=322)', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_tarp_ecp_pdbbind.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('PDBBind group TARP data not yet available — run group_eval job first.')

In [ ]:
# 2b: Three-panel ECP — PoseBusters
if pb_names_tarp is not None and all(pb_tarp[g] is not None for g in GROUPS):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    for ax, g in zip(axes, GROUPS):
        f = pb_tarp[g]
        ecp, alpha = ecp_from_fractions(f)
        boot = bootstrap_ecp(f, n_bootstrap=500)
        n_valid = np.sum(np.any(np.isfinite(f), axis=1))
        plot_ecp(ecp, alpha, ax=ax, label=f'{LABELS[g]}\n(n={n_valid})', color=COLORS[g], bootstrap_ecps=boot)
        ax.set_title(LABELS[g], fontsize=11)
    fig.suptitle('Per-group TARP ECP — DiffDock on PoseBusters benchmark (n=303)', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_tarp_ecp_posebusters.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('PoseBusters group TARP data not yet available.')

In [ ]:
# 2c: Cross-dataset overlay — one panel per group, PDBBind vs PoseBusters
datasets_ready = (
    pdb_names_tarp is not None and all(pdb_tarp[g] is not None for g in GROUPS) and
    pb_names_tarp is not None  and all(pb_tarp[g] is not None for g in GROUPS)
)
if datasets_ready:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    for ax, g in zip(axes, GROUPS):
        for ds_name, f, ls in [('PDBBind', pdb_tarp[g], '-'), ('PoseBusters', pb_tarp[g], '--')]:
            ecp, alpha = ecp_from_fractions(f)
            n_valid = np.sum(np.any(np.isfinite(f), axis=1))
            ax.plot(alpha, ecp, ls=ls, color=COLORS[g], lw=2, label=f'{ds_name} (n={n_valid})')
        ax.plot([0, 1], [0, 1], 'k:', lw=1)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
        ax.set_xlabel('Credibility level α')
        ax.set_ylabel('Expected coverage probability')
        ax.set_title(LABELS[g])
        ax.legend(fontsize=9)
    fig.suptitle('Per-group TARP: PDBBind vs PoseBusters', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_tarp_ecp_overlay.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Need both datasets to produce overlay — data not yet available.')

In [ ]:
# 2d: Per-complex mean-f histograms — calibration signature per complex
# mean_f ≈ 0.5 → calibrated; > 0.5 → overconfident; < 0.5 → underconfident
if pdb_names_tarp is not None and all(pdb_tarp[g] is not None for g in GROUPS):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, g in zip(axes, GROUPS):
        f = pdb_tarp[g]  # (N, K)
        mean_f = np.nanmean(f, axis=1)
        valid = mean_f[np.isfinite(mean_f)]
        ax.hist(valid, bins=30, color=COLORS[g], edgecolor='white', alpha=0.85)
        ax.axvline(0.5, color='k', ls='--', lw=1.5, label='Perfect cal. (0.5)')
        ax.axvline(np.mean(valid), color='red', ls=':', lw=1.5, label=f'Mean = {np.mean(valid):.3f}')
        ax.set_xlabel('Mean coverage fraction per complex')
        ax.set_ylabel('Count')
        ax.set_title(LABELS[g])
        ax.legend(fontsize=9)
        pct_over = (valid > 0.5).mean() * 100
        pct_under = (valid < 0.5).mean() * 100
        ax.text(0.02, 0.97, f'Overconf: {pct_over:.1f}%\nUnderconf: {pct_under:.1f}%',
                transform=ax.transAxes, va='top', fontsize=9)
    fig.suptitle('Per-complex mean TARP f — PDBBind', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_tarp_mean_f_hist_pdbbind.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('PDBBind group TARP data not yet available.')

## 3 · Per-Group Distance Distributions

Raw distances from crystal pose for all S=40 DiffDock samples per complex.
- **Translation**: L2 distance between centroids (Å)
- **Rotation**: geodesic angle on SO(3) = arccos((tr(R)-1)/2) (rad)
- **Torsion**: RMS of wrapped angular differences (rad)

Distances are symmetry-corrected via spyrmsd before extracting rotation/torsion.

In [ ]:
# 3a: Distance distributions — histograms of best-of-N vs rank-1 vs all samples
dist_keys = ['translation', 'rotation', 'torsion_rms']
dist_labels = {'translation': 'Translation (Å)', 'rotation': 'Rotation (rad)', 'torsion_rms': 'Torsion RMS (rad)'}

if pdb_names_dist is not None and all(pdb_dist[k] is not None for k in dist_keys):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, key in zip(axes, dist_keys):
        D = pdb_dist[key]  # (N, S)
        rank1 = D[:, 0]       # rank-1 pose (highest-confidence)
        best  = np.nanmin(D, axis=1)  # oracle best-of-N
        flat  = D.flatten()
        flat  = flat[np.isfinite(flat)]
        ax.hist(flat,  bins=50, color='lightgray', density=True, label='All samples', alpha=0.7)
        ax.hist(best,  bins=30, color=COLORS['torsion'] if key == 'torsion_rms' else COLORS[key],
                density=True, alpha=0.7, label=f'Best-of-N (n={D.shape[1]})')
        ax.hist(rank1, bins=30, color='tomato', density=True, alpha=0.7, label='Rank-1')
        ax.set_xlabel(dist_labels[key])
        ax.set_ylabel('Density')
        ax.set_title(dist_labels[key])
        ax.legend(fontsize=9)
        ax.text(0.98, 0.97, f'Rank-1 median: {np.nanmedian(rank1):.3f}\nBest median:   {np.nanmedian(best):.3f}',
                transform=ax.transAxes, ha='right', va='top', fontsize=9)
    fig.suptitle('Per-group distance distributions — PDBBind (n=322)', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_distances_hist_pdbbind.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('PDBBind group distance data not yet available.')

In [ ]:
# 3b: Best-of-N success rate curves (analogous to RMSD < 2Å but for each group)
if pdb_names_dist is not None and all(pdb_dist[k] is not None for k in dist_keys):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    thresholds = {
        'translation': np.linspace(0, 15, 300),
        'rotation':    np.linspace(0, np.pi, 300),
        'torsion_rms': np.linspace(0, np.pi, 300),
    }
    for ax, key in zip(axes, dist_keys):
        D = pdb_dist[key]
        rank1 = D[:, 0]
        best  = np.nanmin(D, axis=1)
        ts = thresholds[key]
        cdf_rank1 = [(np.isfinite(rank1) & (rank1 < t)).mean() for t in ts]
        cdf_best  = [(np.isfinite(best)  & (best  < t)).mean() for t in ts]
        ax.plot(ts, cdf_rank1, color='tomato',    lw=2, label='Rank-1')
        ax.plot(ts, cdf_best,  color=COLORS['torsion'] if key == 'torsion_rms' else COLORS[key],
                lw=2, label=f'Best-of-{D.shape[1]}')
        ax.set_xlabel(dist_labels[key])
        ax.set_ylabel('Fraction of complexes')
        ax.set_title(dist_labels[key])
        ax.set_ylim(0, 1); ax.legend(fontsize=9)
    fig.suptitle('Per-group cumulative success rate — PDBBind', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_success_rate_pdbbind.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('PDBBind group distance data not yet available.')

## 4 · Stratification by Ligand Flexibility

Group complexes by number of rotatable bonds:
- Rigid: 0–2 bonds
- Low: 3–5 bonds
- Medium: 6–9 bonds
- Flexible: ≥10 bonds

Key question: does DiffDock's torsion calibration degrade with more flexible ligands?

In [ ]:
FLEX_BINS   = [0, 3, 6, 10, 999]
FLEX_LABELS = ['Rigid (0–2)', 'Low (3–5)', 'Medium (6–9)', 'Flexible (≥10)']
FLEX_COLORS = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

def flex_bin(n):
    for i in range(len(FLEX_BINS) - 1):
        if FLEX_BINS[i] <= n < FLEX_BINS[i + 1]:
            return i
    return len(FLEX_LABELS) - 1

# ── TARP ECP by flexibility ───────────────────────────────────────────────────
if pdb_names_tarp is not None and pdb_nrot_tarp is not None and all(pdb_tarp[g] is not None for g in GROUPS):
    bins = np.array([flex_bin(n) for n in pdb_nrot_tarp])

    for g in GROUPS:
        fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
        f_all = pdb_tarp[g]
        # Overall in first panel
        ecp, alpha = ecp_from_fractions(f_all)
        plot_ecp(ecp, alpha, ax=axes[0], label=f'All (n={len(f_all)})', color='k')
        axes[0].set_title(f'All complexes')
        # Per-flexibility-bin
        for i, (lbl, col) in enumerate(zip(FLEX_LABELS, FLEX_COLORS)):
            mask = bins == i
            if mask.sum() < 5:
                axes[i + 1].text(0.5, 0.5, f'n={mask.sum()}\n(too few)', ha='center', va='center')
                axes[i + 1].set_title(lbl)
                continue
            f_sub = f_all[mask]
            ecp, alpha = ecp_from_fractions(f_sub)
            boot = bootstrap_ecp(f_sub, n_bootstrap=300)
            plot_ecp(ecp, alpha, ax=axes[i + 1], label=f'{lbl} (n={mask.sum()})', color=col, bootstrap_ecps=boot)
            axes[i + 1].set_title(lbl)
        fig.suptitle(f'{LABELS[g]} TARP by ligand flexibility — PDBBind', fontsize=12, y=1.01)
        plt.tight_layout()
        plt.savefig(f'{FIGURES}/group_tarp_{g}_by_flex_pdbbind.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('PDBBind group TARP + n_rot_bonds data not yet available.')

In [ ]:
# 4b: Overlay all three groups for each flexibility bin
if pdb_names_tarp is not None and pdb_nrot_tarp is not None and all(pdb_tarp[g] is not None for g in GROUPS):
    bins = np.array([flex_bin(n) for n in pdb_nrot_tarp])
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    for i, (lbl, ax) in enumerate(zip(['All'] + FLEX_LABELS, axes)):
        mask = np.ones(len(pdb_nrot_tarp), dtype=bool) if i == 0 else (bins == i - 1)
        for g in GROUPS:
            f_sub = pdb_tarp[g][mask]
            if f_sub.shape[0] < 5:
                continue
            ecp, alpha = ecp_from_fractions(f_sub)
            ax.plot(alpha, ecp, color=COLORS[g], lw=2, label=f'{LABELS[g]} (n={mask.sum()})')
        ax.plot([0, 1], [0, 1], 'k:', lw=1)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
        ax.set_xlabel('Credibility level α'); ax.set_ylabel('ECP')
        ax.set_title(lbl, fontsize=11)
        ax.legend(fontsize=8)
    fig.suptitle('Group TARP by flexibility — all groups overlaid', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_tarp_by_flex_overlay_pdbbind.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Data not yet available.')

In [ ]:
# 4c: Mean f per complex vs n_rot_bonds scatter (one panel per group)
if pdb_names_tarp is not None and pdb_nrot_tarp is not None and all(pdb_tarp[g] is not None for g in GROUPS):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, g in zip(axes, GROUPS):
        f = pdb_tarp[g]
        mean_f = np.nanmean(f, axis=1)
        valid = np.isfinite(mean_f) & np.isfinite(pdb_nrot_tarp.astype(float))
        x = pdb_nrot_tarp[valid]
        y = mean_f[valid]
        ax.scatter(x, y, color=COLORS[g], alpha=0.35, s=15)
        # Running median
        xuniq = np.unique(x)
        med_y = [np.median(y[x == xv]) for xv in xuniq]
        ax.plot(xuniq, med_y, 'k-', lw=2, label='Median per n_rot_bonds')
        ax.axhline(0.5, color='k', ls='--', lw=1, alpha=0.5, label='Perfect cal.')
        r, p = stats.spearmanr(x, y)
        ax.set_xlabel('n_rot_bonds')
        ax.set_ylabel('Mean TARP f')
        ax.set_title(LABELS[g])
        ax.text(0.97, 0.03, f'Spearman r={r:.3f}, p={p:.3f}',
                transform=ax.transAxes, ha='right', va='bottom', fontsize=9)
        ax.legend(fontsize=8)
    fig.suptitle('Mean TARP f vs ligand flexibility — PDBBind', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_tarp_meanf_vs_flex_pdbbind.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Data not yet available.')

## 5 · Stratification by Protein Family

Uses the pre-built `pdb_annotations.csv` protein class annotations.
Same 7-group taxonomy as the existing TARP/MIRA notebook.

In [ ]:
import pandas as pd

ANN_PATH = '../data/pdb_annotations.csv'
if os.path.exists(ANN_PATH):
    ann = pd.read_csv(ANN_PATH)
    ann_map = dict(zip(ann['pdb_id'], ann['protein_class'].fillna('Unknown')))
else:
    print('pdb_annotations.csv not found — protein family analysis unavailable.')
    ann_map = {}

FAM_GROUPS = [
    ('Hydrolases',         lambda c: 'hydrolase'     in c.lower()),
    ('Transferases',       lambda c: 'transferase'   in c.lower()),
    ('Sugar Binding',      lambda c: 'sugar binding' in c.lower()),
    ('Signaling',          lambda c: 'signaling'     in c.lower()),
    ('Transcription/Gene', lambda c: any(k in c.lower() for k in
                                         ('transcription', 'nuclear protein',
                                          'dna binding', 'rna binding', 'gene regulation'))),
    ('Oxidoreductases',    lambda c: 'oxidoreductase' in c.lower()),
    ('Other',              lambda c: True),
]
FAM_NAMES  = [f[0] for f in FAM_GROUPS]
FAM_COLORS = ['C0', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6']

def assign_family(pdb_id):
    pc = ann_map.get(pdb_id, 'Unknown')
    for name, fn in FAM_GROUPS:
        if fn(pc):
            return name
    return 'Other'

print('Family taxonomy loaded.')

In [ ]:
# 5a: ECP curves per family, one figure per group (3 × 7 panels)
if pdb_names_tarp is not None and ann_map and all(pdb_tarp[g] is not None for g in GROUPS):
    fam_labels = np.array([assign_family(n) for n in pdb_names_tarp])

    for g in GROUPS:
        f_all = pdb_tarp[g]
        ncols = 4
        nrows = (len(FAM_NAMES) + 1 + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4.5))
        axes = axes.ravel()

        ecp_all, alpha_all = ecp_from_fractions(f_all)
        plot_ecp(ecp_all, alpha_all, ax=axes[0], label=f'All (n={len(f_all)})', color='k')
        axes[0].set_title('All families')

        for i, (fname, fcolor) in enumerate(zip(FAM_NAMES, FAM_COLORS)):
            mask = fam_labels == fname
            ax = axes[i + 1]
            if mask.sum() < 5:
                ax.text(0.5, 0.5, f'n={mask.sum()}\n(too few)', ha='center', va='center')
                ax.set_title(fname, fontsize=9)
                continue
            f_sub = f_all[mask]
            ecp, alpha = ecp_from_fractions(f_sub)
            boot = bootstrap_ecp(f_sub, n_bootstrap=300)
            plot_ecp(ecp, alpha, ax=ax, label=f'{fname} (n={mask.sum()})', color=fcolor, bootstrap_ecps=boot)
            ax.set_title(f'{fname} (n={mask.sum()})', fontsize=9)

        for j in range(len(FAM_NAMES) + 1, len(axes)):
            axes[j].set_visible(False)

        fig.suptitle(f'{LABELS[g]} TARP by protein family — PDBBind', fontsize=12, y=1.01)
        plt.tight_layout()
        plt.savefig(f'{FIGURES}/group_tarp_{g}_by_family_pdbbind.png', dpi=150, bbox_inches='tight')
        plt.show()
else:
    print('Data or annotations not yet available.')

In [ ]:
# 5b: Summary table — mean f per family per group
if pdb_names_tarp is not None and ann_map and all(pdb_tarp[g] is not None for g in GROUPS):
    fam_labels = np.array([assign_family(n) for n in pdb_names_tarp])
    print(f'{"Family":<22} {"N":>4}  ' + '  '.join(f'{LABELS[g]:>22}' for g in GROUPS))
    print('-' * (22 + 4 + 3 * 24))
    for fname in FAM_NAMES:
        mask = fam_labels == fname
        n = mask.sum()
        row = f'{fname:<22} {n:>4}'
        for g in GROUPS:
            mf = np.nanmean(pdb_tarp[g][mask]) if n > 0 else float('nan')
            direction = '↑ overconf' if mf < 0.45 else ('↓ underconf' if mf > 0.55 else 'calibrated ')
            row += f'  {mf:>8.3f} ({direction})'
        print(row)
    print()
    print('(mean f < 0.5 → overconfident; mean f > 0.5 → underconfident)')

## 6 · Correlation Across Groups

Are the calibration errors correlated? A complex that is overconfident in translation —
does it also tend to be overconfident in rotation/torsion?

In [ ]:
# 6a: Rank-1 distance correlation matrix
if pdb_names_dist is not None and all(pdb_dist[k] is not None for k in dist_keys):
    # Use rank-1 pose distances
    d_tr = pdb_dist['translation'][:, 0]
    d_ro = pdb_dist['rotation'][:, 0]
    d_to = pdb_dist['torsion_rms'][:, 0]

    valid = np.isfinite(d_tr) & np.isfinite(d_ro) & np.isfinite(d_to)
    D = np.column_stack([d_tr[valid], d_ro[valid], d_to[valid]])
    corr = np.corrcoef(D.T)

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xticklabels(['Translation', 'Rotation', 'Torsion'], rotation=30, ha='right')
    ax.set_yticklabels(['Translation', 'Rotation', 'Torsion'])
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center',
                    color='w' if abs(corr[i, j]) > 0.6 else 'k', fontsize=12)
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title('Rank-1 distance correlation across groups\nPDBBind (n={})'.format(valid.sum()))
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_dist_corr_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

    for i, gi in enumerate(['Translation', 'Rotation', 'Torsion']):
        for j, gj in enumerate(['Translation', 'Rotation', 'Torsion']):
            if j > i:
                r, p = stats.spearmanr(D[:, i], D[:, j])
                print(f'{gi} vs {gj}: Spearman r={r:.3f}, p={p:.4f}')
else:
    print('Distance data not yet available.')

In [ ]:
# 6b: Mean-f correlation across groups (TARP perspective)
if pdb_names_tarp is not None and all(pdb_tarp[g] is not None for g in GROUPS):
    mf = {g: np.nanmean(pdb_tarp[g], axis=1) for g in GROUPS}
    valid = np.isfinite(mf['translation']) & np.isfinite(mf['rotation']) & np.isfinite(mf['torsion'])

    pairs = [('translation', 'rotation'), ('translation', 'torsion'), ('rotation', 'torsion')]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (g1, g2) in zip(axes, pairs):
        x = mf[g1][valid]
        y = mf[g2][valid]
        ax.scatter(x, y, color='steelblue', alpha=0.35, s=12)
        ax.axhline(0.5, color='k', ls='--', lw=0.8, alpha=0.5)
        ax.axvline(0.5, color='k', ls='--', lw=0.8, alpha=0.5)
        r, p = stats.spearmanr(x, y)
        ax.set_xlabel(f'Mean f — {LABELS[g1]}')
        ax.set_ylabel(f'Mean f — {LABELS[g2]}')
        ax.set_title(f'{LABELS[g1]} vs {LABELS[g2]}')
        ax.text(0.97, 0.03, f'r={r:.3f}  p={p:.3f}',
                transform=ax.transAxes, ha='right', va='bottom', fontsize=9)
    fig.suptitle('Per-complex TARP mean f: cross-group correlation — PDBBind', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_tarp_meanf_correlation.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Data not yet available.')

In [ ]:
# 6c: Translation vs torsion distance coloured by PoseBusters validity
if (pdb_names_dist is not None and all(pdb_dist[k] is not None for k in dist_keys)
        and pb_validity):
    # PoseBusters validity for rank-1 on PDBBind set
    rank1_valid = np.array([
        'rank1.sdf' in pb_validity.get(n, {}).get('valid_ranks', []) or
        'rank1_confidence-1.sdf' in str(pb_validity.get(n, {}).get('valid_ranks', []))
        for n in pdb_names_dist
    ])

    d_tr = pdb_dist['translation'][:, 0]
    d_to = pdb_dist['torsion_rms'][:, 0]
    valid = np.isfinite(d_tr) & np.isfinite(d_to)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(d_tr[valid & ~rank1_valid], d_to[valid & ~rank1_valid],
               color='tomato', alpha=0.4, s=15, label='PB invalid')
    ax.scatter(d_tr[valid & rank1_valid],  d_to[valid & rank1_valid],
               color='steelblue', alpha=0.5, s=15, label='PB valid')
    ax.set_xlabel('Translation distance (Å)')
    ax.set_ylabel('Torsion RMS distance (rad)')
    ax.set_title('Rank-1 translation vs torsion distance\ncoloured by PoseBusters validity')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_dist_tr_vs_tors_pb_validity.png', dpi=150, bbox_inches='tight')
    plt.show()
    pct_valid = rank1_valid[valid].mean() * 100
    print(f'PB valid rank-1: {pct_valid:.1f}% of {valid.sum()} complexes with distances')
else:
    print('Data not yet available.')

## 7 · Connection to PoseBusters Failures

Does calibration depend on whether the pose passes physical validity checks?
Split complexes into PoseBusters-valid (rank-1 passes all checks) vs invalid.

In [ ]:
# 7a: Mean-f violin plots — PB valid vs invalid (per group)
if pdb_names_tarp is not None and pb_validity and all(pdb_tarp[g] is not None for g in GROUPS):
    # Build PB validity mask aligned to TARP complex names
    def _is_rank1_valid(pdb_id):
        vr = pb_validity.get(pdb_id, {}).get('valid_ranks', [])
        return any('rank1' in r for r in vr)

    pb_mask = np.array([_is_rank1_valid(n) for n in pdb_names_tarp])
    print(f'PB valid: {pb_mask.sum()}, PB invalid: {(~pb_mask).sum()}')

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for ax, g in zip(axes, GROUPS):
        f = pdb_tarp[g]
        mf = np.nanmean(f, axis=1)
        valid_data = mf[pb_mask  & np.isfinite(mf)]
        inv_data   = mf[~pb_mask & np.isfinite(mf)]
        vp = ax.violinplot([inv_data, valid_data], positions=[0, 1],
                           showmedians=True, showextrema=False)
        vp['bodies'][0].set_facecolor('tomato');    vp['bodies'][0].set_alpha(0.6)
        vp['bodies'][1].set_facecolor('steelblue'); vp['bodies'][1].set_alpha(0.6)
        ax.axhline(0.5, color='k', ls='--', lw=1, label='Perfect cal.')
        ax.set_xticks([0, 1])
        ax.set_xticklabels([f'PB invalid\n(n={len(inv_data)})', f'PB valid\n(n={len(valid_data)})'])
        ax.set_ylabel('Mean TARP f')
        ax.set_title(LABELS[g])
        u, p = stats.mannwhitneyu(inv_data, valid_data, alternative='two-sided')
        ax.text(0.97, 0.97, f'Mann-Whitney p={p:.3f}',
                transform=ax.transAxes, ha='right', va='top', fontsize=9)
        ax.legend(fontsize=8)
    fig.suptitle('TARP calibration: PB-valid vs PB-invalid rank-1 — PDBBind', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_tarp_pb_valid_vs_invalid.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Data not yet available.')

In [ ]:
# 7b: ECP curves conditioned on PB validity
if pdb_names_tarp is not None and pb_validity and all(pdb_tarp[g] is not None for g in GROUPS):
    def _is_rank1_valid(pdb_id):
        vr = pb_validity.get(pdb_id, {}).get('valid_ranks', [])
        return any('rank1' in r for r in vr)

    pb_mask = np.array([_is_rank1_valid(n) for n in pdb_names_tarp])

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    for ax, g in zip(axes, GROUPS):
        f = pdb_tarp[g]
        for mask, lbl, col, ls in [
            (np.ones(len(pb_mask), dtype=bool), 'All', 'k',         '-'),
            (pb_mask,                           'PB valid',   'steelblue', '--'),
            (~pb_mask,                          'PB invalid', 'tomato',    ':'),
        ]:
            f_sub = f[mask]
            if f_sub.shape[0] < 5:
                continue
            ecp, alpha = ecp_from_fractions(f_sub)
            ax.plot(alpha, ecp, color=col, ls=ls, lw=2, label=f'{lbl} (n={mask.sum()})')
        ax.plot([0, 1], [0, 1], 'grey', lw=1, ls='dotted')
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
        ax.set_xlabel('Credibility level α')
        ax.set_ylabel('Expected coverage')
        ax.set_title(LABELS[g])
        ax.legend(fontsize=9)
    fig.suptitle('Group TARP conditioned on PoseBusters validity — PDBBind', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_tarp_ecp_by_pb_validity.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Data not yet available.')

In [ ]:
# 7c: Translation distance vs pocket miss (PB minimum_distance_to_protein check)
if pdb_names_dist is not None and pdb_dist['translation'] is not None and pb_validity:
    def _pocket_miss(pdb_id):
        """Returns True if rank-1 failed minimum_distance_to_protein."""
        cfs = pb_validity.get(pdb_id, {}).get('check_failures', {})
        return cfs.get('minimum_distance_to_protein', 0) > 0

    pocket_miss = np.array([_pocket_miss(n) for n in pdb_names_dist])
    d_tr = pdb_dist['translation'][:, 0]
    valid = np.isfinite(d_tr)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    ax = axes[0]
    ax.hist(d_tr[valid & ~pocket_miss], bins=40, color='steelblue', alpha=0.7, density=True,
            label=f'No pocket miss (n={(valid & ~pocket_miss).sum()})')
    ax.hist(d_tr[valid & pocket_miss],  bins=40, color='tomato',    alpha=0.7, density=True,
            label=f'Pocket miss (n={(valid & pocket_miss).sum()})')
    ax.set_xlabel('Rank-1 translation distance (Å)')
    ax.set_ylabel('Density')
    ax.set_title('Translation distance by pocket miss')
    ax.legend(fontsize=9)

    ax = axes[1]
    n_pm   = pocket_miss[valid].sum()
    n_npm  = (~pocket_miss[valid]).sum()
    m_pm   = np.nanmedian(d_tr[valid & pocket_miss])
    m_npm  = np.nanmedian(d_tr[valid & ~pocket_miss])
    ax.bar(['No pocket miss', 'Pocket miss'], [m_npm, m_pm],
           color=['steelblue', 'tomato'], alpha=0.8)
    ax.set_ylabel('Median translation distance (Å)')
    ax.set_title(f'Median translation: {m_npm:.2f} vs {m_pm:.2f} Å')
    u, p = stats.mannwhitneyu(d_tr[valid & ~pocket_miss], d_tr[valid & pocket_miss],
                              alternative='two-sided')
    ax.text(0.5, 0.95, f'Mann-Whitney p={p:.3f}', transform=ax.transAxes,
            ha='center', va='top', fontsize=10)

    fig.suptitle('Translation distance and pocket miss — PDBBind', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_translation_vs_pocket_miss.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Data not yet available.')

## 8 · PoseBusters Benchmark — Group TARP

Repeat key analyses on the PoseBusters benchmark set (303 complexes with newer ligands).
Unlike PDBBind, PoseBusters uses a flat directory layout.

In [ ]:
# 8a: Three-panel ECP — PoseBusters, with flexibility overlay
if (pb_names_tarp is not None and pb_nrot_tarp is not None
        and all(pb_tarp[g] is not None for g in GROUPS)):
    bins_pb = np.array([flex_bin(n) for n in pb_nrot_tarp])

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    for ax, g in zip(axes, GROUPS):
        f = pb_tarp[g]
        ecp, alpha = ecp_from_fractions(f)
        n_valid = np.sum(np.any(np.isfinite(f), axis=1))
        ax.plot(alpha, ecp, color=COLORS[g], lw=2.5, label=f'All (n={n_valid})')
        for bi, (bl, bc) in enumerate(zip(FLEX_LABELS, FLEX_COLORS)):
            mask = bins_pb == bi
            if mask.sum() < 5:
                continue
            ecp_sub, alpha_sub = ecp_from_fractions(f[mask])
            ax.plot(alpha_sub, ecp_sub, color=bc, lw=1, ls='--', alpha=0.7,
                    label=f'{bl} (n={mask.sum()})')
        ax.plot([0, 1], [0, 1], 'grey', lw=1, ls=':')
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect('equal')
        ax.set_xlabel('Credibility level α')
        ax.set_ylabel('ECP')
        ax.set_title(LABELS[g])
        ax.legend(fontsize=7.5)
    fig.suptitle('Per-group TARP with flexibility — DiffDock on PoseBusters (n=303)', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/group_tarp_pb_with_flex.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('PoseBusters group TARP data not yet available.')

In [ ]:
# 8b: Side-by-side summary table: PDBBind vs PoseBusters, mean-f per group
print('Mean TARP f summary (overconfident < 0.5 < underconfident)\n')
print(f'{"":<20}  {"PDBBind":>15}  {"PoseBusters":>15}')
print('-' * 55)
for g in GROUPS:
    pdb_mf = (np.nanmean(pdb_tarp[g]) if pdb_tarp[g] is not None else float('nan'))
    pb_mf  = (np.nanmean(pb_tarp[g])  if pb_tarp[g]  is not None else float('nan'))
    pdb_str = f'{pdb_mf:.4f}' if np.isfinite(pdb_mf) else '(pending)'
    pb_str  = f'{pb_mf:.4f}'  if np.isfinite(pb_mf)  else '(pending)'
    print(f'{LABELS[g]:<20}  {pdb_str:>15}  {pb_str:>15}')